<a href="https://colab.research.google.com/github/ab23ms233/GRISHMA-iitkgp-2026/blob/main/Mechanistic-interpretability/interpretability_intro_to_transformerlens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Keeping terminal alive while idle

In [ ]:
function KeepAlive() {
    console.log("Simulating activity...");
    // Selects the toolbar connect button inside the shadow DOM
    const connectButton = document.querySelector("#top-toolbar > colab-connect-button")
        ?.shadowRoot?.querySelector("#connect");

    if (connectButton) {
        connectButton.click();
    }
}
// Triggers the function every 60 seconds (60000 milliseconds)
setInterval(KeepAlive, 60000);


# Cloning GitHub repository

In [ ]:
!git clone https://github.com/callummcdougall/ARENA_3.0.git

# Setup

In [ ]:
# NumPy expected by TransformerLens
!pip install "numpy<2"

# Correct beartype version
!pip install "beartype<0.15"

# Install TransformerLens with dependencies
!pip install transformer_lens==2.18.0

# Additional packages used in the notebook
!pip install circuitsvis eindex-callum

## Changing directory

In [ ]:
%cd /content/ARENA_3.0/chapter1_transformer_interp/exercises/

## Importing libraries and exercises

In [ ]:
import functools
import sys
from pathlib import Path
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
from eindex import eindex
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
from transformer_lens.hook_points import HookPoint

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part2_intro_to_mech_interp"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_intro_to_mech_interp.tests as tests
from plotly_utils import (
    hist,
    imshow,
    plot_comp_scores,
    plot_logit_attribution,
    plot_loss_difference,
)

# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# PART 1: TransformerLens Introduction

## Importing GPT-2 small

In [ ]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

## Examining architecture

In [ ]:
gpt2_small.cfg

## Calculating loss

We calculate the loss of the model in predicting the next token in the sequence.

1. For every token in the sequence, the model outputs a logit of vectors over the entire vocabulary of size V. The length of tokens is T. The tokens are represented as:
$${x_0, x_1, …, x_{T-1}}$$

2. Suppose we are predicting the token at $t+1$. A softmax function is applied over the logits to output probability distributions:
$$p_t(i) = \frac{e^{z_{t,i}}}{\sum_{j=1}^{V} e^{z_{t,i}}}$$

    where $z_{t,i}$ is the i-th logit for token at t.

3. If the next token is $x_{t+1}$, then loss for prediction:
$$l_t = -log\,p_t(x_{t+1})$$

4. $$L = -\frac{1}{T-1} \sum_{t=0}^{T-2} \log P\left(x_{t+1} \mid x_{\leq t}\right)$$

In [ ]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model out, let's find the loss on this paragraph!"""

loss = gpt2_small(model_description_text, return_type="loss")
print("Model loss:", loss)


## Tokenization

We explore the different methods for tokenization of a text

In [ ]:
print(gpt2_small.to_str_tokens("gpt2"))
print(gpt2_small.to_str_tokens(["gpt2", "gpt2"]))
print(gpt2_small.to_tokens("gpt2"))
print(gpt2_small.to_string([50256, 70, 457, 17]))

## Checking correctly predicted tokens

We calculate what and how many tokens are correctly predicted

In [ ]:
# Calculating logits
logits: Tensor = gpt2_small(model_description_text, return_type="logits")
logits.squeeze().shape

In [ ]:
# Squeeze operation drops all the single-element dimensions. Over here batch dimension is one, hence it is dropped.
# We omit the last prediction since there is no actual word after the last one
prediction = logits.squeeze().argmax(dim=-1)[:-1]

# YOUR CODE HERE - get the model's prediction on the text
# Predictions are after the first token, hence omitting the 1st one
original = gpt2_small.to_tokens(model_description_text).squeeze()[1:]
correct = 0
correct_tokens = []

# Comparing the number of correct tokens
for tokid, origid in zip(prediction, original):
    if tokid == origid:
        # tokid is the id of token, token is the original word
        token = gpt2_small.to_single_str_token(tokid.item())
        correct_tokens.append(token)
        correct += 1

print(f"Number of correct predictions: {correct}/{len(original)}")
print(f"Correctly predicted tokens: {correct_tokens}")

## Checking type of logits and cache

In [ ]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)

print(type(gpt2_logits), type(gpt2_cache))
print(f"Number of tokens: {len(gpt2_tokens.squeeze())}")

## Calculating attention patterns

We calculate the attention patterns generated by the model and compare them with those that we get by from the query, key, and value matrices

In [ ]:
attn_patterns_from_shorthand = gpt2_cache["pattern", 0]
attn_patterns_from_full_name = gpt2_cache["blocks.0.attn.hook_pattern"]

t.testing.assert_close(attn_patterns_from_shorthand, attn_patterns_from_full_name)

In [ ]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]

# YOUR CODE HERE - define `layer0_pattern_from_q_and_k` manually, by manually performing the steps of the attention calculation (dot product, masking, scaling, softmax)

# HINT
# You'll need to use three different cache indexes in all:
# gpt2_cache["pattern", 0] to get the attention patterns, which have shape [nhead, seqQ, seqK]
# gpt2_cache["q", 0] to get the query vectors, which have shape [seqQ, nhead, headsize]
# gpt2_cache["k", 0] to get the key vectors, which have shape [seqK, nhead, headsize]

qvec, kvec = gpt2_cache["q", 0], gpt2_cache["k", 0]
headsize = qvec.shape[-1]
seqQ, seqK = qvec.shape[0], kvec.shape[0]

# Generating a lower triangular matrix with lower values as True and upper values as False
mask = t.tril(t.ones((seqQ, seqK), dtype=t.bool)).to(device)
attn_scores = einops.einsum(qvec, kvec, "seqQ nhead headsize, seqK nhead headsize -> nhead seqQ seqK")/headsize**0.5

# Substitute -infinity wherever mask values are False
# We do not make the upper triangular values zero after calculating the attention pattern since it does not preserve probability
inf = t.tensor(float("inf"))
layer0_pattern_from_q_and_k = t.where(mask, attn_scores, -inf).softmax(dim=-1)

t.testing.assert_close(layer0_pattern_from_cache, layer0_pattern_from_q_and_k)
print("Tests passed!")

In [ ]:
print(qvec.shape, kvec.shape)

## Visualizing attention patterns

We visualize the attention pattern of all the heads of the first layer

In [ ]:
attention_pattern = gpt2_cache["pattern", 0]
print(attention_pattern.shape)
gpt2_str_tokens = gpt2_small.to_str_tokens(gpt2_text)

print("Layer 0 Head Attention Patterns:")
display(
    cv.attention.attention_patterns(
        tokens=gpt2_str_tokens,
        attention=attention_pattern
    )
)


## Visualizing Neuron Activations

We visualize the tokens that a neuron is activated to in different layers

In [ ]:
# Post means the activations after the GELU function of the MLP layer. We stack the activations of all the layers together.
neuron_activations_for_all_layers = t.stack([
    gpt2_cache["post", layer] for layer in range(gpt2_small.cfg.n_layers)
], dim=1)
# shape = (seq_pos, layers, neurons)

cv.activations.text_neuron_activations(
    tokens=gpt2_str_tokens,
    activations=neuron_activations_for_all_layers
)

In [ ]:
neuron_activations_for_all_layers_rearranged = utils.to_numpy(einops.rearrange(neuron_activations_for_all_layers, "seq layers neurons -> 1 layers seq neurons"))

# Displays the top k tokens that a neuron is activated to in each layer
cv.topk_tokens.topk_tokens(
    # Some weird indexing required here ¯\_(ツ)_/¯
    tokens=[gpt2_str_tokens],
    activations=neuron_activations_for_all_layers_rearranged,
    max_k=7,
    first_dimension_name="Layer",
    third_dimension_name="Neuron",
    first_dimension_labels=list(range(12))
)

# PART 2: Finding Induction Heads

## Configuration of toy model

The toy model has only two layers and 12 heads in each layer

In [ ]:
cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)


## Importing toy model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "callummcdougall/attn_only_2L_half"
FILENAME = "attn_only_2L_half.pth"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [ ]:
model = HookedTransformer(cfg)
pretrained_weights = t.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

## Visualizing attention patterns

In [ ]:
text = "My name is ChatGPT. I am a transformer-based Large Language Model. I can do extraordinary stuffs. Try me out and tell me how you feel."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

# YOUR CODE HERE - visualize attention
tokens = model.to_str_tokens(text)

for i in range(model.cfg.n_layers):
    attn_ptn = cache["pattern", i]
    display(
    cv.attention.attention_patterns(
        tokens=tokens,
        attention=attn_ptn
    )
)

## Detecting attention head types

We detect the current token attention head, previous token attention head, and first token attention head

In [ ]:
def detect_attn_heads(cache: ActivationCache, start_row, target_func, false_threshold=0.1) -> list[str]:
    """
    Detects attention heads of a specific type
    """
    # List to store the attention heads
    heads = []
    tot_false_frac = []

    # Iterating through the layers and heads
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            # Attention pattern of current layer and head
            attn_ptn = cache["pattern", layer][head]
            # Tensor of row nums
            rows = t.arange(start_row, attn_ptn.shape[0])
            cols = target_func(rows)
            target_vals = attn_ptn[rows, cols]
            # max of each of the rows
            row_max = attn_ptn[rows].amax(dim=-1)
            false_frac = (target_vals < row_max).float().mean()

            # Append head if fraction of false counts is less than threshold
            if false_frac < false_threshold:
                tot_false_frac.append(false_frac)
                heads.append(f"{layer}.{head}")

    if len(tot_false_frac) == 0:
        frac = 0
    else:
        frac = sum(tot_false_frac)/len(tot_false_frac)
    return (heads, frac)


def current_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be current-token heads
    """
    # row-th col = row-th row
    start_row = 0
    # For curr attn detector, the row-th column contains the greatest value for every row
    target_fn = lambda rows: rows
    curr_heads, false_frac = detect_attn_heads(cache, start_row, target_fn)

    print(f"False frac for current heads: {false_frac}")
    return curr_heads

def prev_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be previous-token heads
    """
    # (row-1)-th col = row-th row
    start_row = 1
    # For curr attn detector, the (row-1)-th column contains the greatest value for every row
    target_fn = lambda rows: rows-1
    prev_heads, false_frac = detect_attn_heads(cache, start_row, target_fn)

    print(f"False frac for previous heads: {false_frac}")
    return prev_heads

def first_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be first-token heads
    """
    # row-th col = row-th row
    start_row = 0
    # For curr attn detector, the row-th column contains the greatest value for every row
    target_fn = lambda rows: t.zeros_like(rows)
    first_heads, false_frac = detect_attn_heads(cache, start_row, target_fn)

    print(f"False frac for first heads: {false_frac}")
    return first_heads


print("Heads attending to current token  = ", ", ".join(current_attn_detector(cache)))
print("Heads attending to previous token = ", ", ".join(prev_attn_detector(cache)))
print("Heads attending to first token    = ", ", ".join(first_attn_detector(cache)))


## Generating repeated tokens

We write a function to generate repeated token sequences, train the model on such sequences and calculate loss.

In [ ]:
def generate_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> Int[Tensor, "batch_size full_seq_len"]:
    """
    Generates a sequence of repeated random tokens

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
    """
    t.manual_seed(0)  # for reproducibility
    # Beginning of sentence token
    prefix = (t.ones(batch_size, 1) * model.tokenizer.bos_token_id).long()
    # Generating the first sequence of tokens
    tokens = t.randint(0, model.cfg.d_vocab, (batch_size, seq_len), dtype=t.long)
    # Concatenating the sequence to form repeated tokens
    rep_tokens = t.concat((prefix, tokens, tokens), dim=-1).to(device)
    return rep_tokens


def run_and_cache_model_repeated_tokens(
    model: HookedTransformer, seq_len: int, batch_size: int = 1
) -> tuple[Tensor, Tensor, ActivationCache]:
    """
    Generates a sequence of repeated random tokens, and runs the model on it, returning (tokens,
    logits, cache). This function should use the `generate_repeated_tokens` function above.

    Outputs are:
        rep_tokens: [batch_size, 1+2*seq_len]
        rep_logits: [batch_size, 1+2*seq_len, d_vocab]
        rep_cache: The cache of the model run on rep_tokens
    """
    rep_tokens = generate_repeated_tokens(model, seq_len, batch_size)
    rep_logits, rep_cache = model.run_with_cache(rep_tokens)

    return (rep_tokens, rep_logits, rep_cache)

# Caclulate the loss of the model per token
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    logprobs = logits.log_softmax(dim=-1)
    # We want to get logprobs[b, s, tokens[b, s+1]], in eindex syntax this looks like:
    correct_logprobs = eindex(logprobs, tokens, "b s [b s+1]")
    return correct_logprobs


seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = -get_log_probs(rep_logits, rep_tokens).squeeze()

print(f"Performance on the first half: {log_probs[:seq_len].mean():.3f}")
print(f"Performance on the second half: {log_probs[seq_len:].mean():.3f}")

plot_loss_difference(log_probs, rep_str, seq_len)

As can be seen from the plot, the loss in the second half is clearly less than the the first.

## Visualizing induction heads

These are the attention heads in the 2nd layer which retain memory of pairs of tokens which occured together earlier in the sequence. If the same token occurs again, it is able to predict the next token after it.

In [ ]:
for i in range(model.cfg.n_layers):
    attn_ptn = rep_cache["pattern", i]
    display(
    cv.attention.attention_patterns(
        tokens=rep_str,
        attention=attn_ptn
    )
)

## Identifying induction heads

These identify the next token for tokens which have occured together earlier in the sequence. We write a function to identify induction heads.

In [ ]:
def induction_attn_detector(cache: ActivationCache) -> list[str]:
    """
    Returns a list e.g. ["0.2", "1.4", "1.9"] of "layer.head" which you judge to be induction heads

    Remember - the tokens used to generate rep_cache are (bos_token, *rand_tokens, *rand_tokens)
    """
    seq_len = (cache["pattern", 0][0].shape[0]-1)//2    # The induction pattern is only visible in the 2nd half of the repetitive text
    false_threshold = 0.2   # If the false counts are below this threshold then it is an induction head.
    start_row = seq_len+1
    induc_heads, false_frac = detect_attn_heads(cache, )
    # Traversing through each layer and head
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            # Attention pattern for a head in a layer
            attn_ptn = cache["pattern", layer][head]


            # 1st pos - BOS token
            # (1 - seq_len+1) pos - seq_len tokens
            # (seq_len+1 - 2*seq_len+1) - seq_len tokens
            # The query token (y-axis, destination) receives value from the key token (x-axis, source). The query token in the 2nd half receives answer from the key token which appeared after it in the 1st half - (row, row-seq_len+1)
            rows =
            for row in range(seq_len+1, attn_ptn.shape[0]):
                max_row = max(attn_ptn[row])

                # If the criteria for induction head is not met in the row
                if attn_ptn[row, row-seq_len+1] < max_row:
                    false_count += 1

            # If false count is below threshold then it is an induction head
            if false_count/attn_ptn.shape[0] < false_threshold:
                induc_heads.append(f"{layer}.{head}")

    return induc_heads

print("Induction heads = ", ", ".join(induction_attn_detector(rep_cache)))


# PART 3: Hooks

## Induction score for heads

We write a function which calculates the induction score for each head in each layer.

In [ ]:
seq_len = 50
batch_size = 10
rep_tokens_10 = generate_repeated_tokens(model, seq_len, batch_size)

# We make a tensor to store the induction score for each head.
# We put it on the model's device to avoid needing to move things between the GPU and CPU,
# which can be slow.
induction_score_store = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)


def induction_score_hook(pattern: Float[Tensor, "batch head_index dest_pos source_pos"], hook: HookPoint):
    """
    Calculates the induction score, and stores it in the [layer, head] position of the
    `induction_score_store` tensor.
    """
    # We are taking the offseted diagonal of the attention matrix for each batch and head (batch n_heads query key -> batch n_heads diag)
    diag = pattern.diagonal(dim1=-2, dim2=-1, offset=1-seq_len)
    # We are calculating the mean of diag elements over all batches for a head. (batch n_heads diag -> n_heads)
    induction_score = einops.reduce(diag, "batch head_index position -> head_index", "mean")
    # Storing the result
    induction_score_store[hook.layer(), :] = induction_score


# We make a boolean filter on activation names, that's true only on attention pattern names
pattern_hook_names_filter = lambda name: name.endswith("pattern")

# Run with hooks (this is where we write to the `induction_score_store` tensor`)
model.run_with_hooks(
    rep_tokens_10,
    return_type=None,  # For efficiency, we don't need to calculate the logits
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)],
)

# Plot the induction scores for each head in each layer
imshow(
    induction_score_store,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head",
    text_auto=".2f",
    width=1200,
    height=500,
)

From the graph it is visible that heads 1.4 and 1.10 are the main induction heads, as we found out in the previous sections.

## Visualizing induction heads in GPT-2 small

In [ ]:
# This plots the induction heads in each layer
def visualize_pattern_hook(
    pattern: Float[Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    print("Layer: ", hook.layer())
    display(cv.attention.attention_patterns(tokens=gpt2_small.to_str_tokens(rep_tokens[0]), attention=pattern.mean(0)))


# YOUR CODE HERE - find induction heads in gpt2_small
induction_score_store = t.zeros((gpt2_small.cfg.n_layers, gpt2_small.cfg.n_heads), device=model.cfg.device)
gpt2_small.run_with_hooks(rep_tokens_10, return_type=None, fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)])

In [ ]:
# Plot the induction scores for each head in each layer
imshow(
    induction_score_store,
    labels={"x": "Head", "y": "Layer"},
    title="Induction Score by Head",
    text_auto=".2f",
    width=1200,
    height=500,
)

In [ ]:
induction_head_layers = []
threshold = 0.5

# Identifying induction heads
for layer in range(induction_score_store.shape[0]):
    if (induction_score_store[layer] >= threshold).any():
        induction_head_layers.append(layer)

# Plotting them
fwd_hooks = [(utils.get_act_name("pattern", layer), visualize_pattern_hook) for layer in induction_head_layers]

gpt2_small.run_with_hooks(rep_tokens, return_type=None, fwd_hooks=fwd_hooks)


## Logit attribution

We calculate the contribution from each component (embedding and layers) of the model for calculating the next token. We multiply the output from the component directly with the unembedding matrix.

In [ ]:
def logit_attribution(
    embed: Float[Tensor, "seq d_model"],
    l1_results: Float[Tensor, "seq nheads d_model"],
    l2_results: Float[Tensor, "seq nheads d_model"],
    W_U: Float[Tensor, "d_model d_vocab"],
    tokens: Int[Tensor, "seq"],
) -> Float[Tensor, "seq-1 n_components"]:
    """
    Inputs:
        embed: the embeddings of the tokens (i.e. token + position embeddings)
        l1_results: the outputs of the attention heads at layer 1 (with head as one of the dims)
        l2_results: the outputs of the attention heads at layer 2 (with head as one of the dims)
        W_U: the unembedding matrix
        tokens: the token ids of the sequence

    Returns:
        Tensor of shape (seq_len-1, n_components)
        represents the concatenation (along dim=-1) of logit attributions from:
            the direct path (seq-1,1)
            layer 0 logits (seq-1, n_heads)
            layer 1 logits (seq-1, n_heads)
        so n_components = 1 + 2*n_heads
    """
    W_U_correct_tokens = W_U[:, tokens[1:]]

    # Direct attributions from embeddings
    direct_attributions = einops.einsum(W_U_correct_tokens, embed[:-1], "emb seq, seq emb -> seq")
    # Attributions from 1st layer
    l1_attributions = einops.einsum(W_U_correct_tokens, l1_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    # Attributions from 2nd layer
    l2_attributions = einops.einsum(W_U_correct_tokens, l2_results[:-1], "emb seq, seq nhead emb -> seq nhead")
    return t.concat([direct_attributions.unsqueeze(-1), l1_attributions, l2_attributions], dim=-1)


text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."
logits, cache = model.run_with_cache(text, remove_batch_dim=True)
str_tokens = model.to_str_tokens(text)
tokens = model.to_tokens(text)

with t.inference_mode():
    embed = cache["embed"]
    l1_results = cache["result", 0]
    l2_results = cache["result", 1]
    logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, tokens[0])
    # Uses fancy indexing to get a len(tokens[0])-1 length tensor, where the kth entry is the predicted logit for the correct k+1th token
    correct_token_logits = logits[0, t.arange(len(tokens[0]) - 1), tokens[0, 1:]]
    t.testing.assert_close(logit_attr.sum(1), correct_token_logits, atol=1e-3, rtol=0)
    print("Tests passed!")


In [ ]:
# Visualizing the logit attributions
embed = cache["embed"]
l1_results = cache["result", 0]
l2_results = cache["result", 1]
logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, tokens.squeeze())

plot_logit_attribution(model, logit_attr, tokens, title="Logit attribution (demo prompt)")

The tokens with very high logit attribution are the ones which are the first token in common bigrams. For instance, the highest contribution on the direct path comes from | manip|, because this is very likely to be followed by |ulative|.

Layer 1 heads have higher contribution than layer 0. This is because they read the data encoded by layer 0 from the residual stream to make correct predictions.

In [ ]:
# YOUR CODE HERE - plot logit attribution for the induction sequence (i.e. using `rep_tokens` and
# `rep_cache`), and interpret the results.
embed = rep_cache["embed"]
l1_results = rep_cache["result", 0]
l2_results = rep_cache["result", 1]
logit_attr = logit_attribution(embed, l1_results, l2_results, model.W_U, rep_tokens.squeeze())

plot_logit_attribution(model, logit_attr, rep_tokens, title="Logit attribution for induction heads")

## Ablations

We can make the output from a head in a layer equal to 0 and check the increase in loss. This tells us the importance of that head in calculating the output.

In [ ]:
# Make the output of the head zero
def head_zero_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
) -> None:
    z[:, :, head_index_to_ablate, :] = 0


def get_ablation_scores(
    model: HookedTransformer,
    tokens: Int[Tensor, "batch seq"],
    ablation_function: Callable = head_zero_ablation_hook,
) -> Float[Tensor, "n_layers n_heads"]:
    """
    Returns a tensor of shape (n_layers, n_heads) containing the increase in cross entropy loss
    from ablating the output of each head.
    """
    # Initialize an object to store the ablation scores
    ablation_scores = t.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

    # Calculating loss without any ablation, to act as a baseline
    model.reset_hooks()
    seq_len = (tokens.shape[1] - 1) // 2
    logits = model(tokens, return_type="logits")
    loss_no_ablation = -get_log_probs(logits, tokens)[:, -(seq_len - 1) :].mean()

    for layer in tqdm(range(model.cfg.n_layers)):
        for head in range(model.cfg.n_heads):
            # Extracting name of z activation for layer
            act_z = utils.get_act_name("z", layer)
            # zero_ablation function where head_index is pre-defined, so that head_index does not have to be passed as an argument
            zero_ablation = functools.partial(ablation_function, head_index_to_ablate=head)
            # Calculating logits after ablating head output
            ablated_logits = model.run_with_hooks(tokens, return_type="logits", fwd_hooks=[(act_z, zero_ablation)])
            # Calculating loss with ablation
            loss_with_ablation = -get_log_probs(ablated_logits, tokens)[:, -(seq_len - 1) :].mean()
            # Calculating difference in losses
            loss_diff = loss_with_ablation-loss_no_ablation
            # Storing result
            ablation_scores[layer, head] = loss_diff

    return ablation_scores


ablation_scores = get_ablation_scores(model, rep_tokens)
tests.test_get_ablation_scores(ablation_scores, model, rep_tokens)


In [ ]:
imshow(ablation_scores, labels={"x": "Head", "y": "Layer", "color":"Logit diff"}, title="Loss difference after ablating heads", text_auto="0.2f", width=1200, height=500)


We see that **head 0.7** has a large ablation score. This is expected as we saw that it is a previous token head in the earlier section. **Heads 1.4** and **1.10** also have a high ablation score since they are induction heads.

In logit attribution, we did not see the importance of head 0.7 since it was not directly contributing to the output, but helping the other heads by reading what it had written to the residual stream. From ablations, we saw that head 0.7 also has a significant role, since the other heads do not function properly without it.

In logit attribution we just see the result of a normal forward pass (observation). The ablation method changes the output of the model during a forward pass. This is known as **causal intervention**. This helps to see which heads contribute both directly (through inference) and indirectly (by helping other heads).

In [ ]:
# Mean ablation over batch dimension for head
def head_mean_ablation_hook(
    z: Float[Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index_to_ablate: int,
) -> None:
    z[:, :, head_index_to_ablate, :] = z[:, :, head_index_to_ablate, :].mean(dim=0)


rep_tokens_batch = run_and_cache_model_repeated_tokens(model, seq_len=50, batch_size=10)[0]
mean_ablation_scores = get_ablation_scores(model, rep_tokens_batch, ablation_function=head_mean_ablation_hook)

imshow(
    mean_ablation_scores,
    labels={"x": "Head", "y": "Layer", "color": "Logit diff"},
    title="Loss Difference After Ablating Heads",
    text_auto=".2f",
    width=900,
    height=350,
)


# PART 4: Reverse-engineering induction circuits

Till now we saw that:

1. Head 0.7 is a previous token head.
2. Heads 1.4 and 1.10 are induction heads.

Some things to note:

1. $$Attention\,pattern\,(AP) = softmax \left(\frac{QK^T}{\sqrt{d_{model}}} \right)$$

This is known as the **QK circuit**.

2. $$Attention\,output = AP \cdot V \cdot W_O$$

This is known as the **OV circuit**.

3. Composition means one head's output is used by another head's input.

    a. **Q-composition**: When Head A's output is used by Head B for computing queries.

    b. **K-composition**: Head B uses Head A's o/p for calculating keys.

4. Head 0.7 is a previous token head. It comes entirely from its QK circuit. It writes a copy of previous token in a different subspace.

    a. Suppose the sentence is "Alice likes". It copies the information (embedding vector) that "Alice is the previous token of likes" into the residual stream.

    b. Alice = $e_1$ (embedding vector).
    Previous token = Alice: $e_2$ (another embedding vector). Both point in different directions.

6. Output of 0.7 is used by key input of 1.10 by K-composition.

    a. Suppose a sequence is: A B C A B. When 1.10 is at A B |, it asks at what position earlier, B was the previous token (query).
    
    b. It receives $e_2$ (hidden representation of Alice made by 0.7 - key input) - K-composition.

    c. It does not copy $e_2$. Instead, it performs $e_1W_V$ to obtain the actual representation of the token "Alice". Hence, no V-composition.




## Calculating different properties of Factored matrices

In [ ]:
A = t.randn(5, 2)
B = t.randn(2, 5)
AB = A @ B
AB_factor = FactoredMatrix(A, B)
print("Norms:")
print(f"Usual method: {AB.norm()}")
print(f"Factored matrix method: {AB_factor.norm()}")

print(f"Right dim: {AB_factor.rdim}, Left dim: {AB_factor.ldim}, Hidden dim: {AB_factor.mdim}")

In [ ]:
print("Eigenvalues:")
print(f"Torch method: {t.linalg.eig(AB).eigenvalues}")
print(f"Factored matrix method: {AB_factor.eigenvalues}")

print("\nSingular Values:")
print(t.linalg.svd(AB).S)
print(AB_factor.S)

print("\nFull SVD:")
print(AB_factor.svd())

## Building the OV circuit

The OV circuit can be written as:

$$W_EW_V^hW_O^hW_U = W_EW_{OV}^hW_U$$

where h is the notation of head. For example, h = 1.4 means layer = 1 and head = 4.

The dimensions of the matrices are:

1. $W_E - [d_{vocab}, d_{model}]$
2. $W_V - [d_{model}, d_{head}]$
3. $W_O - [d_{head}, d_{model}]$
4. $W_U - [d_{model}, d_{vocab}]$
5. $W_{OV} - [d_{model}, d_{model}]$

Hence, the multiplication gives the final dimension as - $[d_{vocab}, d_{vocab}]$

In [ ]:
head_index = 4
layer = 1

# YOUR CODE HERE - complete the `full_OV_circuit` object
W_E = model.W_E
W_O = model.W_O[layer, head_index]
W_V = model.W_V[layer, head_index]
W_U = model.W_U

full_OV_circuit = FactoredMatrix(W_E@W_V, W_O@W_U)
tests.test_full_OV_circuit(full_OV_circuit, model, layer, head_index)

In [ ]:
indices = t.randint(0, model.cfg.d_vocab, (200,))
full_OV_circuit_sample = full_OV_circuit[indices, indices].AB

imshow(
    full_OV_circuit_sample,
    labels={"x": "Logits on output token", "y": "Input token"},
    title="Full OV circuit for copying head",
    width=700,
    height=600,
)

The matrix obtained is between the tokens received from the **QK circuit** and the tokens received after the **OV circuit**.

The QK circuit determines which token to attend to and the OV circuit extracts the embedding of that token. This increases the logit of that token, giving us a diagonal matrix.

In [ ]:
def top_1_acc(full_OV_circuit: FactoredMatrix, batch_size: int = 1000) -> float:
    """
    Return the fraction of the time that the maximum value is on the circuit diagonal.
    """
    count = 0
    # We divide the entire vocab matrix into matrices of 1000 rows (1 batch) at a time and compute how many times the max element lies on the diagonal in every batch
    # Indices is a batch of numbers like 0-999, 1000-2000, etc
    for indices in tqdm(t.split(t.arange(full_OV_circuit.shape[0], device=device), batch_size)):
        # Slice having the ith 1000 rows from the full matrix
        AB_slice = full_OV_circuit[indices].AB
        # t.argmax(AB_slice, dim=-1) - (indices of columns where the max element lies along a row) == indices - (max element lies along the diagonal)
        # Number of positions where the maximum element lies on the diagonal
        num_diag_batch = (t.argmax(AB_slice, dim=-1) == indices).float().sum().item()
        count += num_diag_batch

    return count/full_OV_circuit.shape[0]

print(f"\nFraction of time that the best logit is on diagonal: {top_1_acc(full_OV_circuit):.4f}")

1. For an (m, n) matrix,
    $$Rank \le  min(m, n)$$

2. Rank theorem for matrix multiplication:
    $$rank(AB) \le min(rank(A), rank(B))$$

3. $$W_{OV} = W_VW_O \\
\implies rank(W_{OV}) \le min(rank(W_V), rank(W_O)) \le d_{head}$$

    It has only $d_{head}$ independent dimensions to represent the text.

4. Rank theorem for addition:
    $$rank(A + B) \le rank(A) + rank(B)$$

5. For 2 heads,
    $$W_{OV} = W_V^{h_1}W_O^{h_1} + W_V^{h_2}W_O^{h_2} \\
    \implies rank(W_{OV}) \le 2d_{head}$$

    It has $2d_{head}$ dimensions to represent the text.

In [ ]:
head_indices = [4, 10]
layer = 1

# YOUR CODE HERE - compute the effective OV circuit, and run `top_1_acc` on it
W_V_both = einops.rearrange(model.W_V[layer, head_indices], "head d_model d_head -> d_model (head d_head)")
W_O_both = einops.rearrange(model.W_O[layer, head_indices], "head d_head d_model -> (head d_head) d_model")

effective_OV_circuit = W_E @ FactoredMatrix(W_V_both, W_O_both) @ W_U
print(f"\nFraction of time that the best logit is on diagonal: {top_1_acc(effective_OV_circuit):.4f}")

## QK previous-token circuit of Head 0.7

$$QK\,circuit - W_{pos}W_Q^{0.7}(W_K^{0.7})^TW_{pos}^T$$

The dimensions are:
1. $W_{pos} - [n_{ctx}, d_{model}]$
2. $W_Q, W_K - [d_{model}, d_{head}]$

The multiplication gives us a dimension of $[n_{ctx}, n_{ctx}]$, where $n_{ctx}$ is the maximum context length of a sequence.

In [ ]:
layer = 0
head_index = 7

# Compute full QK matrix (for positional embeddings)
W_pos = model.W_pos
W_QK = model.W_Q[layer, head_index] @ model.W_K[layer, head_index].T
pos_by_pos_scores = W_pos @ W_QK @ W_pos.T

# Mask, scale and softmax the scores
mask = t.tril(t.ones_like(pos_by_pos_scores)).bool()
pos_by_pos_pattern = t.where(mask, pos_by_pos_scores / model.cfg.d_head**0.5, -1.0e6).softmax(-1)

# Plot the results
print(f"Avg lower-diagonal value: {pos_by_pos_pattern.diag(-1).mean():.4f}")
imshow(
    utils.to_numpy(pos_by_pos_pattern[:200, :200]),
    labels={"x": "Key", "y": "Query"},
    title="Attention patterns for prev-token QK circuit, first 200 indices",
    width=700,
    height=600,
)

(i,j)-th element is representing the attention score paid by the i-th token to the j-th token. This should be very large when j=i−1 (and smaller for all other values of j), because this is a previous head token.

Why is it justified to ignore token encodings? In this case, it turns out that the positional encodings have a much larger effect on the attention scores than the token encodings.

## K composition

$$
x W_Q^{1.H}
=
\left(
e + pe + \sum_{h=0}^{11} x^{0.h}
\right) W_Q^{1.H}
=
e W_Q^{1.H}
+
pe\, W_Q^{1.H}
+
\sum_{h=0}^{11} x^{0.h} W_Q^{1.H}
$$

where,

$x$ - final residual stream after layer 0

$x^{0.h}$ - outputs after head h in layer 0

e - token embedding

pe - positional encoding

Hence, $x$ can be written as a sum of 14 tensors having dimension `[seq_len, d_model]`

In [ ]:
def decompose_qk_input(cache: ActivationCache) -> Float[Tensor, "n_heads+2 posn d_model"]:
    """
    Retrieves all the input tensors to the first attention layer, and concatenates them along the
    0th dim.

    The [i, :, :]th element is y_i (from notation above). The sum of these tensors along the 0th
    dim should be the input to the first attention layer.
    """
    embed = cache["embed"].unsqueeze(0)
    pos_embed = cache["pos_embed"].unsqueeze(0)
    attn_out = einops.rearrange(cache["result", 0], "pos n_heads d_model -> n_heads pos d_model")

    return t.concat([embed, pos_embed, attn_out], dim=0)


def decompose_q(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of query vectors for each decomposed QK input.

    The [i, :, :]th element is y_i @ W_Q (so the sum along axis 0 is just the q-values).
    """
    layer = 1
    W_Q = model.W_Q[layer, ind_head_index]

    q_vec = einops.einsum(decomposed_qk_input, W_Q, "comp pos d_model, d_model d_head -> comp pos d_head")
    return q_vec


def decompose_k(
    decomposed_qk_input: Float[Tensor, "n_heads+2 posn d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[Tensor, "n_heads+2 posn d_head"]:
    """
    Computes the tensor of key vectors for each decomposed QK input.

    The [i, :, :]th element is y_i @ W_K(so the sum along axis 0 is just the k-values)
    """
    layer = 1
    W_Q = model.W_K[layer, ind_head_index]

    k_vec = einops.einsum(decomposed_qk_input, W_Q, "comp pos d_model, d_model d_head -> comp pos d_head")
    return k_vec


# Recompute rep tokens/logits/cache, if we haven't already
seq_len = 50
batch_size = 1
(rep_tokens, rep_logits, rep_cache) = run_and_cache_model_repeated_tokens(model, seq_len, batch_size)
rep_cache.remove_batch_dim()

ind_head_index = 4

# First we get decomposed q and k input, and check they're what we expect
decomposed_qk_input = decompose_qk_input(rep_cache)
decomposed_q = decompose_q(decomposed_qk_input, ind_head_index, model)
decomposed_k = decompose_k(decomposed_qk_input, ind_head_index, model)
t.testing.assert_close(
    decomposed_qk_input.sum(0),
    rep_cache["resid_pre", 1] + rep_cache["pos_embed"],
    rtol=0.01,
    atol=1e-05,
)
t.testing.assert_close(decomposed_q.sum(0), rep_cache["q", 1][:, ind_head_index], rtol=0.01, atol=0.001)
t.testing.assert_close(decomposed_k.sum(0), rep_cache["k", 1][:, ind_head_index], rtol=0.01, atol=0.01)

# Second, we plot our results
component_labels = ["Embed", "PosEmbed"] + [f"0.{h}" for h in range(model.cfg.n_heads)]
for decomposed_input, name in [(decomposed_q, "query"), (decomposed_k, "key")]:
    imshow(
        utils.to_numpy(decomposed_input.pow(2).sum([-1])),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        width=800,
        height=400,
    )

In [ ]:
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: HookedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]

    The [i, j, 0, 0]th element is y_i @ W_QK @ y_j^T (so the sum along both first axes are the
    attention scores)
    """
    return einops.einsum(decomposed_q, decomposed_k, "q_comp q_pos d_head, k_comp k_pos d_head -> q_comp k_comp q_pos k_pos")/model.cfg.d_head**0.5


tests.test_decompose_attn_scores(decompose_attn_scores, decomposed_q, decomposed_k, model)

In [ ]:
# First plot: attention score contribution from (query_component, key_component) = (Embed, L0H7), you can replace this
# with any other pair and see that the values are generally much smaller, i.e. this pair dominates the attention score
# calculation
decomposed_scores = decompose_attn_scores(decomposed_q, decomposed_k, model)

q_label = "Embed"
k_label = "0.6"
decomposed_scores_from_pair = decomposed_scores[component_labels.index(q_label), component_labels.index(k_label)]

imshow(
    utils.to_numpy(t.tril(decomposed_scores_from_pair)),
    title=f"Attention score contributions from query = {q_label}, key = {k_label}<br>(by query & key sequence positions)",
    width=700,
)


# Second plot: std dev over query and key positions, shown by component. This shows us that the other pairs of
# (query_component, key_component) are much less important, without us having to look at each one individually like we
# did in the first plot!
decomposed_stds = einops.reduce(
    decomposed_scores, "query_decomp key_decomp query_pos key_pos -> query_decomp key_decomp", t.std
)
imshow(
    utils.to_numpy(decomposed_stds),
    labels={"x": "Key Component", "y": "Query Component"},
    title="Std dev of attn score contributions across sequence positions<br>(by query & key comp)",
    x=component_labels,
    y=component_labels,
    width=700,
)

In [ ]:
def find_K_comp_full_circuit(
    model: HookedTransformer, prev_token_head_index: int, ind_head_index: int
) -> FactoredMatrix:
    """
    Returns a (vocab, vocab)-size FactoredMatrix, with the first dimension being the query side
    (direct from token embeddings) and the second dimension being the key side (going via the
    previous token head).
    """
    W_E = model.W_E
    W_QK = FactoredMatrix(model.W_Q[1, ind_head_index], model.W_K[1, ind_head_index].T)
    W_OV = FactoredMatrix(model.W_V[0, prev_token_head_index], model.W_O[0, prev_token_head_index])

    return (W_E @ W_QK @ W_OV.T @ W_E.T)


prev_token_head_index = 7
ind_head_index = 4
K_comp_circuit = find_K_comp_full_circuit(model, prev_token_head_index, ind_head_index)

tests.test_find_K_comp_full_circuit(find_K_comp_full_circuit, model)

print(f"Token frac where max-activating key = same token: {top_1_acc(K_comp_circuit.T):.4f}")


In [ ]:
for layer in list(rep_cache.keys()):
    print(f"{layer}: {rep_cache[layer].shape}")

In [ ]:
rep_cache["result", 0].shape

In [ ]:
model.W_Q[1]

In [ ]:
arr = t.randn((3, 3))
arr

In [ ]:
max_rows = t.argmax(arr, dim=-1)
max_rows